# Long-Form Bengali ASR — Dual T4 GPU Parallel Inference

### Strategy
- **2 model instances**: one on `cuda:0`, one on `cuda:1`
- **ThreadPoolExecutor** with 2 workers — each worker owns one GPU
- **VAD** runs on CPU (avoids cross-GPU tensor conflicts)
- **Identical inference pipeline** to the original — output is unchanged
- **~2× throughput** because two audio files are processed simultaneously

## 1. Installation

In [1]:
!pip install -q torch torchaudio
!pip install -q transformers>=4.30.0
!pip install -q librosa
!pip install -q silero-vad
!pip install -q soundfile
!pip install -q accelerate
!pip install -q pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 93.8 MB/s eta 0:00:00


## 2. Imports & GPU Check

In [2]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from glob import glob
import os
import librosa
import torch
import torchaudio
from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor
import soundfile as sf
import warnings
import re
from collections import Counter
import gc
import csv
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

# ── GPU info ──────────────────────────────────────────────────────────────────
NUM_GPUS = torch.cuda.device_count()
print(f"PyTorch version : {torch.__version__}")
print(f"GPUs available  : {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    mem_gb = props.total_memory / 1e9
    print(f"  GPU {i}: {props.name}  |  {mem_gb:.1f} GB")

if NUM_GPUS < 2:
    print("\n⚠️  Only 1 GPU found — will run on cuda:0 only (no parallelism).")
    GPUS = ["cuda:0"]
else:
    GPUS = ["cuda:0", "cuda:1"]
    print(f"\n✅ Dual-GPU mode enabled — using {GPUS}")

PyTorch version : 2.9.0+cu126
GPUs available  : 2
  GPU 0: Tesla T4  |  15.6 GB
  GPU 1: Tesla T4  |  15.6 GB

✅ Dual-GPU mode enabled — using ['cuda:0', 'cuda:1']


## 3. Configuration

In [3]:
CONFIG = {
    # Model
    'model_name': '/kaggle/input/datasets/aurchichowdhury/finetuned-best-model/best_model',
    'language': 'bn',

    # VAD
    'vad_threshold': 0.5,
    'min_speech_duration_ms': 250,
    'min_silence_duration_ms': 500,
    'speech_pad_ms': 100,

    # Chunking / inference
    'chunk_length_s': 30,
    'max_new_tokens': 440,
    'batch_size': 8,

    # Whisper hallucination guards
    'condition_on_previous_text': False,
    'temperature': 0.0,
    'compression_ratio_threshold': 2.4,
    'logprob_threshold': -1.0,
    'no_speech_threshold': 0.6,

    # Repetition detection
    'max_ngram_repeat': 5,
    'ngram_size': 5,

    # Paths
    'audio_dir': '/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio',
    'test_file': 'test_001.wav',
    'output_dir': '/kaggle/working/transcriptions/',
}
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print("Config OK")

Config OK


## 4. VAD Setup (CPU — safe for multi-threaded use)

In [4]:
# VAD runs on CPU so both GPU workers can call it without device conflicts.
print("Loading Silero VAD on CPU...")
model_vad, vad_utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    onnx=False
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = vad_utils
model_vad = model_vad.to('cpu')

# VAD is not thread-safe out of the box — use a lock
_vad_lock = threading.Lock()

print("✅ VAD model loaded on CPU")

Loading Silero VAD on CPU...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
✅ VAD model loaded on CPU


In [5]:
def apply_vad(audio_path, return_segments=True):
    """
    Apply Silero VAD to an audio file.
    Audio tensor is kept on CPU to match the VAD model device.
    A lock is used so two threads don't call VAD simultaneously.
    """
    wav = read_audio(audio_path, sampling_rate=16000)  # returns CPU tensor
    if not isinstance(wav, torch.Tensor):
        wav = torch.tensor(wav)
    wav = wav.to('cpu')

    with _vad_lock:
        speech_timestamps = get_speech_timestamps(
            wav,
            model_vad,
            threshold=CONFIG['vad_threshold'],
            min_speech_duration_ms=CONFIG['min_speech_duration_ms'],
            min_silence_duration_ms=CONFIG['min_silence_duration_ms'],
            speech_pad_ms=CONFIG['speech_pad_ms'],
            return_seconds=True
        )

    if not speech_timestamps:
        print("  ⚠️  No speech detected")
        return [], wav

    if return_segments:
        return speech_timestamps, wav
    else:
        speech_audio = collect_chunks(speech_timestamps, wav)
        return speech_audio, wav

## 5. Smart Chunking

In [6]:
def merge_vad_segments(speech_timestamps, max_chunk_duration=30.0, overlap_duration=1.0):
    """Merge VAD segments into ~30 s chunks (WhisperX Cut & Merge style)."""
    if not speech_timestamps:
        return []

    merged = []
    cur = {'start': speech_timestamps[0]['start'], 'end': speech_timestamps[0]['end']}

    for seg in speech_timestamps[1:]:
        if seg['end'] - cur['start'] <= max_chunk_duration:
            cur['end'] = seg['end']
        else:
            merged.append(cur.copy())
            cur = {
                'start': max(cur['end'] - overlap_duration, seg['start']),
                'end': seg['end']
            }

    merged.append(cur)
    return merged

## 6. Load Two Pipeline Instances (one per GPU)

In [7]:
def build_pipeline(device_str: str):
    """Load the ASR pipeline on the specified device."""
    print(f"  Loading pipeline on {device_str} ...")
    p = pipeline(
        "automatic-speech-recognition",
        model=CONFIG['model_name'],
        device=device_str,
        torch_dtype=torch.float16,
    )
    print(f"  ✅ Pipeline ready on {device_str}")
    return p

print("Loading ASR pipelines — this may take a few minutes ...\n")

# Build one pipeline per available GPU
# We do this sequentially to avoid OOM during simultaneous weight loading
pipelines = {gpu: build_pipeline(gpu) for gpu in GPUS}

print(f"\n✅ {len(pipelines)} pipeline(s) loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading ASR pipelines — this may take a few minutes ...

  Loading pipeline on cuda:0 ...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

  ✅ Pipeline ready on cuda:0
  Loading pipeline on cuda:1 ...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

  ✅ Pipeline ready on cuda:1

✅ 2 pipeline(s) loaded.


## 7. Hallucination Detection & Post-processing
*(identical to original)*

In [8]:
def detect_repetitions(text, ngram_size=5, max_repeats=5):
    words = text.split()
    if len(words) < ngram_size:
        return False, 0
    ngrams = [' '.join(words[i:i+ngram_size]) for i in range(len(words) - ngram_size + 1)]
    counts = Counter(ngrams)
    max_count = max(counts.values()) if counts else 0
    return max_count > max_repeats, max_count


def remove_repetitions(text, ngram_size=5):
    words = text.split()
    if len(words) < ngram_size * 2:
        return text
    cleaned, i = [], 0
    while i < len(words):
        if i + ngram_size * 2 <= len(words):
            cur  = ' '.join(words[i:i+ngram_size])
            nxt  = ' '.join(words[i+ngram_size:i+ngram_size*2])
            if cur == nxt:
                i += ngram_size
                continue
        cleaned.append(words[i])
        i += 1
    return ' '.join(cleaned)


HALLUCINATION_PATTERNS = []   # add Bengali patterns here if needed

def remove_hallucinations(text):
    cleaned = text
    for pattern in HALLUCINATION_PATTERNS:
        if pattern.lower() in cleaned.lower():
            sentences = cleaned.split('।')
            sentences = [s for s in sentences if pattern.lower() not in s.lower()]
            cleaned = '।'.join(sentences)
    return cleaned.strip()


def post_process_transcription(text):
    text = remove_repetitions(text, ngram_size=CONFIG['ngram_size'])
    text = remove_hallucinations(text)
    text = ' '.join(text.split())
    return text

## 8. Core Transcription Function
*(same logic as original — only `pipe` is passed as an argument)*

In [9]:
def transcribe_long_audio(audio_path, pipe, device_str, use_vad=True, verbose=True):
    """
    Transcribe long-form audio.

    Parameters
    ----------
    audio_path  : str   — path to .wav file
    pipe        : HuggingFace pipeline — pre-loaded on `device_str`
    device_str  : str   — e.g. 'cuda:0' (used only for logging)
    use_vad     : bool
    verbose     : bool
    """
    tag = f"[{device_str}]"
    if verbose:
        print(f"\n{'='*60}")
        print(f"{tag} Processing: {os.path.basename(audio_path)}")
        print(f"{'='*60}")

    try:
        audio_info = sf.info(audio_path)
        duration_mins = audio_info.duration / 60
        if verbose:
            print(f"{tag} Duration: {duration_mins:.2f} min | SR: {audio_info.samplerate} Hz")

        # ── Step 1: VAD ────────────────────────────────────────────────────
        if use_vad:
            if verbose:
                print(f"\n{tag} Step 1: VAD ...")
            speech_timestamps, wav = apply_vad(audio_path, return_segments=True)
            if not speech_timestamps:
                return {'text': '', 'warning': 'No speech detected', 'chunks': []}

            # ── Step 2: Merge into chunks ──────────────────────────────────
            if verbose:
                print(f"{tag} Step 2: Merging segments ...")
            chunks = merge_vad_segments(
                speech_timestamps,
                max_chunk_duration=CONFIG['chunk_length_s'],
                overlap_duration=1.0
            )
            if verbose:
                print(f"{tag} → {len(chunks)} chunks from {len(speech_timestamps)} VAD segments")
        else:
            wav, _ = librosa.load(audio_path, sr=16000)
            wav = torch.tensor(wav)
            duration = len(wav) / 16000
            chunks = [
                {'start': i, 'end': min(i + CONFIG['chunk_length_s'], duration)}
                for i in range(0, int(duration), CONFIG['chunk_length_s'])
            ]

        # ── Step 3: Transcribe each chunk ──────────────────────────────────
        if verbose:
            print(f"\n{tag} Step 3: Transcribing {len(chunks)} chunks ...")

        all_transcriptions = []
        chunk_details      = []

        for idx, chunk in enumerate(tqdm(chunks, desc=f"{tag} Chunks", leave=False)):
            start_s = int(chunk['start'] * 16000)
            end_s   = int(chunk['end']   * 16000)
            chunk_audio = wav[start_s:end_s]

            # Convert to float32 numpy (pipeline expects numpy array)
            if isinstance(chunk_audio, torch.Tensor):
                chunk_audio = chunk_audio.cpu().numpy()
            if chunk_audio.dtype != np.float32:
                chunk_audio = chunk_audio.astype(np.float32)

            try:
                result = pipe(
                    chunk_audio,
                    max_new_tokens=CONFIG['max_new_tokens'],
                    chunk_length_s=CONFIG['chunk_length_s'],
                )
                text = result['text'].strip()

                is_rep, rep_cnt = detect_repetitions(
                    text,
                    ngram_size=CONFIG['ngram_size'],
                    max_repeats=CONFIG['max_ngram_repeat']
                )
                if is_rep:
                    if verbose:
                        print(f"\n{tag} ⚠️  Chunk {idx+1}: repetition (×{rep_cnt}) — cleaning")
                    text = remove_repetitions(text, ngram_size=CONFIG['ngram_size'])

                if text:
                    all_transcriptions.append(text)
                    chunk_details.append({
                        'chunk_id': idx + 1,
                        'start': chunk['start'],
                        'end': chunk['end'],
                        'text': text,
                        'repetitive': is_rep
                    })

            except Exception as e:
                print(f"\n{tag} ❌ Chunk {idx+1} error: {e}")
                continue

            # Periodic GPU cache flush
            if (idx + 1) % 10 == 0:
                torch.cuda.empty_cache()
                gc.collect()

        # ── Step 4: Post-process ───────────────────────────────────────────
        if verbose:
            print(f"\n{tag} Step 4: Post-processing ...")

        full_text    = ' '.join(all_transcriptions)
        cleaned_text = post_process_transcription(full_text)

        if verbose:
            print(f"{tag} ✅ Done! {len(full_text)} → {len(cleaned_text)} chars")

        return {
            'text': cleaned_text,
            'raw_text': full_text,
            'chunks': chunk_details,
            'duration_mins': duration_mins,
            'num_chunks': len(chunks),
        }

    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'text': '', 'error': str(e), 'chunks': []}

## 9. Single-file Test (GPU 0)

In [10]:
# test_audio_path = os.path.join(CONFIG['audio_dir'], CONFIG['test_file'])
# print(f"🎯 Testing on: {CONFIG['test_file']}\n")

# result = transcribe_long_audio(
#     test_audio_path,
#     pipe=pipelines[GPUS[0]],
#     device_str=GPUS[0],
#     use_vad=True,
#     verbose=True
# )

# print(f"\n{'='*60}")
# print("TEST RESULTS")
# print(f"{'='*60}")
# print(f"Duration      : {result.get('duration_mins', 0):.2f} min")
# print(f"Chunks        : {result.get('num_chunks', 0)}")
# print(f"Transcript len: {len(result['text'])} chars")
# print(f"\nPreview (first 500 chars):")
# print(result['text'][:500])
# if len(result['text']) > 500:
#     print("...")

## 10. Batch Processing with Dual-GPU Parallelism

**How it works**
- Audio files are split into two halves (or more groups if you have more GPUs).
- A `ThreadPoolExecutor` with 2 workers runs both halves concurrently.
- Worker 0 always uses `cuda:0`; Worker 1 always uses `cuda:1`.
- Results are collected in order and written to `submission.csv`.

In [11]:
audio_files = sorted(glob(os.path.join(CONFIG['audio_dir'], '*.wav')))
print(f"Found {len(audio_files)} audio files\n")
for i, f in enumerate(audio_files, 1):
    print(f"  {i:3d}. {os.path.basename(f)}")

Found 24 audio files

    1. test_001.wav
    2. test_002.wav
    3. test_003.wav
    4. test_004.wav
    5. test_005.wav
    6. test_006.wav
    7. test_008.wav
    8. test_009.wav
    9. test_010.wav
   10. test_011.wav
   11. test_012.wav
   12. test_013.wav
   13. test_016.wav
   14. test_018.wav
   15. test_019.wav
   16. test_020.wav
   17. test_021.wav
   18. test_022.wav
   19. test_023.wav
   20. test_024.wav
   21. test_027.wav
   22. test_029.wav
   23. test_030.wav
   24. test_032.wav


In [12]:
# ──────────────────────────────────────────────────────────────────────────────
# Worker function — each thread keeps its GPU assignment fixed
# ──────────────────────────────────────────────────────────────────────────────
def process_file(audio_path: str, gpu_id: int) -> dict:
    """
    Transcribe one audio file on the GPU identified by `gpu_id`.
    Returns a dict with keys: filename, file_id, transcript, duration_mins, num_chunks.
    """
    device_str = GPUS[gpu_id % len(GPUS)]
    pipe       = pipelines[device_str]
    filename   = os.path.basename(audio_path)
    file_id    = os.path.splitext(filename)[0]

    try:
        result = transcribe_long_audio(
            audio_path,
            pipe=pipe,
            device_str=device_str,
            use_vad=True,
            verbose=False
        )
        transcript = result.get('text', '')
        print(f"  ✅ [{device_str}] {filename} → {len(transcript)} chars")
        return {
            'filename'     : filename,
            'file_id'      : file_id,
            'transcript'   : transcript,
            'duration_mins': result.get('duration_mins', 0),
            'num_chunks'   : result.get('num_chunks', 0),
            'error'        : None,
        }
    except Exception as e:
        print(f"  ❌ [{device_str}] {filename} → ERROR: {e}")
        return {
            'filename'  : filename,
            'file_id'   : file_id,
            'transcript': '',
            'error'     : str(e),
        }


# ──────────────────────────────────────────────────────────────────────────────
# Submit all files to the thread pool
# ──────────────────────────────────────────────────────────────────────────────
N_WORKERS = len(GPUS)   # 2 for dual T4, 1 as fallback
print(f"Starting batch processing with {N_WORKERS} parallel worker(s) ...\n")

start_time = time.time()

# We use a list to accumulate futures in submission order
futures_ordered = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    for idx, audio_path in enumerate(audio_files):
        gpu_id = idx % N_WORKERS   # round-robin GPU assignment
        future = executor.submit(process_file, audio_path, gpu_id)
        futures_ordered.append((audio_path, future))

    # Collect results as they finish (in submission order below)
    results_map = {}
    for audio_path, future in futures_ordered:
        results_map[audio_path] = future.result()   # blocks until done

# Rebuild results in original file order
results = [results_map[ap] for ap in audio_files]

elapsed = time.time() - start_time

# ──────────────────────────────────────────────────────────────────────────────
# Write submission.csv
# ──────────────────────────────────────────────────────────────────────────────
with open('submission.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['filename', 'transcript'])
    for r in results:
        writer.writerow([r['file_id'], r['transcript']])

print(f"\n{'='*60}")
print("BATCH PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"Total files   : {len(audio_files)}")
print(f"Successful    : {sum(1 for r in results if r['transcript'])}")
print(f"Failed        : {sum(1 for r in results if not r['transcript'])}")
print(f"Total time    : {elapsed/60:.1f} min")
print(f"Avg / file    : {elapsed/max(len(audio_files),1):.1f} s")
print(f"\n✅ Saved to submission.csv")

Starting batch processing with 2 parallel worker(s) ...



[cuda:1] Chunks:   0%|          | 0/84 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom lo

[cuda:0] Chunks:   0%|          | 0/85 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignor

...

[cuda:1] Chunks:   0%|          | 0/83 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignor

  ✅ [cuda:0] test_030.wav → 20211 chars


Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignor

  ✅ [cuda:1] test_032.wav → 19135 chars

BATCH PROCESSING COMPLETE
Total files   : 24
Successful    : 24
Failed        : 0
Total time    : 186.4 min
Avg / file    : 466.0 s

✅ Saved to submission.csv


## 11. Verify Submission

In [13]:
df = pd.read_csv('submission.csv')

print("Submission Summary")
print(f"{'='*60}")
print(f"Total rows             : {len(df)}")
print(f"Columns                : {list(df.columns)}")
print(f"Empty transcripts      : {(df['transcript'].fillna('') == '').sum()}")
print(f"Avg transcript length  : {df['transcript'].str.len().mean():.0f} chars")
print(f"\nFirst 5 rows:")
print(df.head())

print(f"\n{'='*60}")
print("Sample Transcripts")
print(f"{'='*60}")
for i in range(min(3, len(df))):
    row = df.iloc[i]
    preview = str(row['transcript'])[:150]
    if len(str(row['transcript'])) > 150:
        preview += '...'
    print(f"\n{i+1}. {row['filename']}")
    print(f"   Length : {len(str(row['transcript']))} chars")
    print(f"   Preview: {preview}")

Submission Summary
Total rows             : 24
Columns                : ['filename', 'transcript']
Empty transcripts      : 0
Avg transcript length  : 26818 chars

First 5 rows:
   filename                                         transcript
0  test_001  এক্সকিউর এটা নিতে পারে সেটা আপনাকে বেশ ভালো মা...
1  test_002  কিন্তু ইচ্ছাধারী নাগিন সরি নাগ আমি কিন্তু ছবল ...
2  test_003  গল্পটির সত্য অনন্দ পাবলিশারস প্রাইভেট লিমিটেড ...
3  test_004  যেকোনো জায়গায় যেতে রাতের ট্রেনই আমাদের সবচাই...
4  test_005  মিছিন নিবেদন ফ্লাই ডে ক্লাসিক্স ওরে বাবা ওরে ব...

Sample Transcripts

1. test_001
   Length : 19224 chars
   Preview: এক্সকিউর এটা নিতে পারে সেটা আপনাকে বেশ ভালো মানাবে এমনি থেকে ভীষণ সুন্দর মানুষ আপনি যেটা পড়বেন সেটাই ভালো লাগবে তাই হুম থ্যাঙ্ক ইউ ওয়ার্কাম আপনি জানতা...

2. test_002
   Length : 22842 chars
   Preview: কিন্তু ইচ্ছাধারী নাগিন সরি নাগ আমি কিন্তু ছবল দিলে তোমরা একদম বাচ্চ হয়ে যাবে একদম পুরুষ এই ভাই বাঁচান ভাই বাঁচান ভাই ভাই আয়া ঠিক আছে তোমরা ওঠো ওঠো দাঁ...

3. test_003
   Le

In [14]:
import unicodedata


ZW = r"[\u200B-\u200D\uFEFF]"

def normalize_bn_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    s = re.sub(ZW, "", s)
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split())
    return s


class BengaliTranscriptCleaner:
    """
    A comprehensive cleaner for Bengali ASR transcripts.
    Removes hallucinations, artifacts, repetitions, and normalizes text.
    """
    
    def __init__(self):
        # English hallucinations from YouTube training data
        self.english_hallucinations = [
            "thanks for watching",
            "thank you for watching",
            "please subscribe",
            "like and subscribe",
            "subtitles by",
            "transcript by",
            "amara.org community",
            "the amara org community",
            "subtitles by the amara.org community",
            "subtitles by the amara org community",
            "emily beynon",
            "transcript emily beynon",
            "www.amara.org",
            "www.zenga.tv",
            "© bh media group",
            "© bbc",
            "music",
            "applause",
            "laughter",
        ]
        
        # Video artifacts and subtitle symbols
        self.video_artifacts = [
            "♪♪♪",
            "♪",
            "©",
            "®",
            "™",
            "[music]",
            "[applause]",
            "[laughter]",
            "(music)",
            "(applause)",
            "(laughter)",
        ]
        
        # Unwanted symbols
        self.unwanted_symbols = [
            "<>",
            "<<",
            ">>",
            "|||",
            "---",
            "...",
            "…",
        ]
        
        # Bengali-specific normalizations
        self.bengali_normalizations = {
            '\u200b': '',  # Zero-width space
            '\u200c': '',  # Zero-width non-joiner
            '\u200d': '',  # Zero-width joiner
            '\ufeff': '',  # Zero-width no-break space (BOM)
            '\u00a0': ' ',  # Non-breaking space → regular space
            
            # Punctuation spacing
            ' ।': '।',
            ' ,': ',',
            ' .': '.',
            
            # Dash normalization
            '–': '-',  # En dash
            '—': '-',  # Em dash
            '―': '-',  # Horizontal bar
        }
    
    def normalize_bengali_text(self, text: str) -> str:
        """
        Apply Bengali-specific character and spacing normalizations.
        
        Args:
            text: Input text
            
        Returns:
            Normalized text
        """
        # Character replacement
        for old_char, new_char in self.bengali_normalizations.items():
            text = text.replace(old_char, new_char)
        
        # Remove spaces before danda
        text = re.sub(r'\s+।', '।', text)
        
        # Normalize spaces after danda
        text = re.sub(r'।\s+', '। ', text)
        
        # Multiple spaces to single space
        text = re.sub(r'\s{2,}', ' ', text)
        
        return text
    
    def remove_repetitions(self, text: str, ngram_size: int = 5, max_repeat: int = 3) -> str:
        """
        Remove n-gram repetitions (looping patterns).
        
        Args:
            text: Input text
            ngram_size: Size of n-gram to check
            max_repeat: Maximum allowed repetitions before cleaning
            
        Returns:
            Text with repetitions removed
        """
        words = text.split()
        if len(words) < ngram_size:
            return text
        
        cleaned_words = []
        i = 0
        
        while i < len(words):
            if i + ngram_size > len(words):
                cleaned_words.extend(words[i:])
                break
            
            # Get current n-gram
            current_ngram = tuple(words[i:i + ngram_size])
            
            # Count consecutive repetitions
            repeat_count = 1
            j = i + ngram_size
            
            while j + ngram_size <= len(words):
                next_ngram = tuple(words[j:j + ngram_size])
                if next_ngram == current_ngram:
                    repeat_count += 1
                    j += ngram_size
                else:
                    break
            
            # If excessive repetitions, keep only one
            if repeat_count > max_repeat:
                cleaned_words.extend(words[i:i + ngram_size])
                i = j
            else:
                cleaned_words.append(words[i])
                i += 1
        
        return ' '.join(cleaned_words)
    
    def remove_word_level_repetitions(self, text: str, max_repeat: int = 5) -> str:
        """
        Remove single word repetitions.
        
        Args:
            text: Input text
            max_repeat: Maximum allowed repetitions
            
        Returns:
            Text with word repetitions reduced
        """
        words = text.split()
        if not words:
            return text
        
        cleaned_words = []
        i = 0
        
        while i < len(words):
            current_word = words[i]
            count = 1
            
            # Count consecutive identical words
            while i + count < len(words) and words[i + count] == current_word:
                count += 1
            
            # If excessive repetitions, reduce to 2
            if count > max_repeat:
                cleaned_words.extend([current_word] * 2)
            else:
                cleaned_words.extend([current_word] * count)
            
            i += count
        
        return ' '.join(cleaned_words)
    
    def remove_hallucination_patterns(self, text: str) -> str:
        """
        Remove predefined hallucination patterns and artifacts.
        
        Args:
            text: Input text
            
        Returns:
            Text with hallucinations removed
        """
        # Remove English hallucinations (case-insensitive)
        for hallucination in self.english_hallucinations:
            pattern = re.compile(re.escape(hallucination), re.IGNORECASE)
            text = pattern.sub('', text)
        
        # Remove video artifacts
        for artifact in self.video_artifacts:
            text = text.replace(artifact, '')
        
        # Remove unwanted symbols
        for symbol in self.unwanted_symbols:
            text = text.replace(symbol, '')
        
        return text
    
    def remove_urls_and_emails(self, text: str) -> str:
        """
        Remove URLs and email addresses.
        
        Args:
            text: Input text
            
        Returns:
            Text with URLs and emails removed
        """
        # HTTP/HTTPS URLs
        text = re.sub(
            r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+',
            '',
            text
        )
        
        # www. URLs
        text = re.sub(r'www\.[\w\.-]+\.\w+', '', text)
        
        # Email addresses
        text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
        
        return text
    
    def remove_excessive_punctuation(self, text: str) -> str:
        """
        Normalize punctuation marks.
        
        Args:
            text: Input text
            
        Returns:
            Text with normalized punctuation
        """
        # Multiple periods
        text = re.sub(r'\.{3,}', '.', text)
        
        # Multiple question marks
        text = re.sub(r'\?{2,}', '?', text)
        
        # Multiple exclamation marks
        text = re.sub(r'!{2,}', '!', text)
        
        # Multiple commas
        text = re.sub(r',{2,}', ',', text)
        
        # Standalone punctuation
        text = re.sub(r'\s+[.,;:!?]\s+', ' ', text)
        
        return text
    
    def remove_non_bengali_scripts(self, text: str, keep_english: bool = True) -> str:
        """
        Filter out non-Bengali scripts using Unicode ranges.
        
        Args:
            text: Input text
            keep_english: Whether to keep English letters and digits
            
        Returns:
            Filtered text
        """
        if keep_english:
            # Keep Bengali, English, digits, and basic punctuation
            pattern = r'[^\u0980-\u09FFa-zA-Z0-9\s.,;:!?\-\'\"]'
        else:
            # Keep only Bengali and basic punctuation
            pattern = r'[^\u0980-\u09FF\s.,;:!?\-\'\"]'
        
        text = re.sub(pattern, '', text)
        return text
    
    def normalize_whitespace(self, text: str) -> str:
        """
        Clean up whitespace.
        
        Args:
            text: Input text
            
        Returns:
            Text with normalized whitespace
        """
        # Multiple spaces to single
        text = re.sub(r'\s+', ' ', text)
        
        # Remove leading/trailing whitespace
        text = text.strip()
        
        return text
    
    def remove_short_gibberish(self, text: str, min_words: int = 3) -> str:
        """
        Filter out very short, likely meaningless transcripts.
        
        Args:
            text: Input text
            min_words: Minimum word count threshold
            
        Returns:
            Empty string if gibberish, otherwise original text
        """
        words = text.split()
        
        if len(words) < min_words:
            # Find Bengali character sequences
            bengali_chars = re.findall(r'[\u0980-\u09FF]+', text)
            
            # If no Bengali or very short Bengali content
            if not bengali_chars or len(''.join(bengali_chars)) < 5:
                return ""
        
        return text
    
    def remove_timestamps(self, text: str) -> str:
        """
        Remove timestamp patterns from subtitles.
        
        Args:
            text: Input text
            
        Returns:
            Text with timestamps removed
        """
        # [HH:MM:SS] or [MM:SS] format
        text = re.sub(r'\[\d{1,2}:\d{2}(?::\d{2})?\]', '', text)
        
        # (HH:MM:SS) or (MM:SS) format
        text = re.sub(r'\(\d{1,2}:\d{2}(?::\d{2})?\)', '', text)
        
        return text
    
    def remove_filler_words(self, text: str, aggressive: bool = False) -> str:
        """
        Remove conversational filler words in Bengali.
        
        Args:
            text: Input text
            aggressive: Whether to use aggressive filler removal
            
        Returns:
            Text with fillers removed
        """
        if aggressive:
            # Remove repeated fillers
            text = re.sub(r'\b(আ|উম|এই|ও|হুম)\s+\1\b', r'\1', text)
            
            # Multiple "আ" → single
            text = re.sub(r'(\bআ\b\s*){3,}', 'আ ', text)
            
            # Multiple "উম" → single
            text = re.sub(r'(\bউম\b\s*){3,}', 'উম ', text)
        
        return text
    
    def normalize_punctuation_spacing(self, text: str) -> str:
        """
        Normalize spacing around punctuation marks.
        
        Args:
            text: Input text
            
        Returns:
            Text with normalized punctuation spacing
        """
        # Remove spaces before punctuation
        text = re.sub(r'\s+([।,.!?;:])', r'\1', text)
        
        # Add space after punctuation (if missing, not at end)
        text = re.sub(r'([।,.!?;:])([^\s।,.!?;:\d])', r'\1 \2', text)
        
        # Fix multiple consecutive punctuation
        text = re.sub(r'([।,.!?])\1+', r'\1', text)
        
        return text
    
    def clean_text(self, text: str, aggressive: bool = False) -> str:
        """
        Main cleaning pipeline - orchestrates all cleaning methods.
        
        Args:
            text: Input text
            aggressive: Whether to use aggressive cleaning mode
            
        Returns:
            Cleaned text
        """
        if pd.isna(text) or str(text).strip() == "":
            return ""
        
        text = str(text)
        
        # Step 0: Unicode NFC normalization
        # Ensures visually identical Bengali characters (e.g. ড়, ঢ়, য়) use
        # consistent precomposed forms, matching the competition's eval pipeline.
        text = unicodedata.normalize("NFC", text)
        
        # Step 1: Remove timestamps
        text = self.remove_timestamps(text)
        
        # Step 2: Remove URLs and emails
        text = self.remove_urls_and_emails(text)
        
        # Step 3: Remove hallucination patterns
        text = self.remove_hallucination_patterns(text)
        
        # Step 4: Bengali-specific normalization
        text = self.normalize_bengali_text(text)
        
        # Step 5: Remove filler words
        text = self.remove_filler_words(text, aggressive=aggressive)
        
        # Step 6: Remove word-level repetitions
        if aggressive:
            text = self.remove_word_level_repetitions(text, max_repeat=3)
        else:
            text = self.remove_word_level_repetitions(text, max_repeat=5)
        
        # Step 7: Remove n-gram repetitions (5-grams, then 3-grams)
        text = self.remove_repetitions(text, ngram_size=5, max_repeat=2)
        text = self.remove_repetitions(text, ngram_size=3, max_repeat=2)
        
        # Step 8: Clean up punctuation
        text = self.remove_excessive_punctuation(text)
        
        # Step 9: Normalize punctuation spacing
        text = self.normalize_punctuation_spacing(text)
        
        # Step 10: Remove non-Bengali scripts
        if aggressive:
            text = self.remove_non_bengali_scripts(text, keep_english=False)
        else:
            text = self.remove_non_bengali_scripts(text, keep_english=True)
        
        # Step 11: Normalize whitespace
        text = self.normalize_whitespace(text)
        
        # Step 12: Remove short gibberish (aggressive only)
        if aggressive:
            text = self.remove_short_gibberish(text, min_words=3)
        
        # Final step: match competition's exact NFC normalization pipeline
        text = normalize_bn_text(text)
        
        return text

In [15]:
# ── Apply cleaning pipeline to submission.csv ────────────────────────────────
df_sub = pd.read_csv('submission.csv')
cleaner = BengaliTranscriptCleaner()

original_transcripts = df_sub['transcript'].copy()

df_sub['transcript'] = df_sub['transcript'].apply(
    lambda x: cleaner.clean_text(x, aggressive=False)
)

df_sub.to_csv('submission.csv', index=False)

# Stats
original_empty = original_transcripts.isna().sum() + (original_transcripts.astype(str).str.strip() == '').sum()
cleaned_empty = (df_sub['transcript'].str.strip() == '').sum()

print(f"Cleaning complete!")
print(f"  Original empty: {original_empty}")
print(f"  After cleaning empty: {cleaned_empty}")
print(f"  Removed by cleaning: {cleaned_empty - original_empty}")
print(f"\nsubmission.csv overwritten with cleaned transcripts.")
print(df_sub.head())

Cleaning complete!
  Original empty: 0
  After cleaning empty: 0
  Removed by cleaning: 0

submission.csv overwritten with cleaned transcripts.
   filename                                         transcript
0  test_001  এক্সকিউর এটা নিতে পারে সেটা আপনাকে বেশ ভালো মা...
1  test_002  কিন্তু ইচ্ছাধারী নাগিন সরি নাগ আমি কিন্তু ছবল ...
2  test_003  গল্পটির সত্য অনন্দ পাবলিশারস প্রাইভেট লিমিটেড ...
3  test_004  যেকোনো জায়গায় যেতে রাতের ট্রেনই আমাদের সবচাই...
4  test_005  মিছিন নিবেদন ফ্লাই ডে ক্লাসিক্স ওরে বাবা ওরে ব...
